# NES-VMC: 使用 TotalAnsatz 求解扩展系统基态

本 notebook 实现：
1. 使用 TotalAnsatz（行列式形式的波函数）
2. 用扩展哈密顿量训练得到扩展系统的基态
3. 采样计算局域能量矩阵
4. 对角化得到原系统的激发态能量

## 1. 导入必要的库

In [ ]:
import jax
import jax.numpy as jnp
import netket as nk
import netket.experimental as nkx
import numpy as np
from pyscf import gto, scf, fci
from flax import nnx
import optax
from tqdm import tqdm

# 设置 JAX 浮点精度
jax.config.update("jax_enable_x64", True)

print(f"NetKet version: {nk.__version__}")
print(f"JAX version: {jax.__version__}")

## 2. 设置分子系统（H₂ 分子）

In [ ]:
# H₂ 分子定义
bond_length = 1.4  # Bohr
geometry = [('H', (0., 0., 0.)), ('H', (bond_length, 0., 0.))]
mol = gto.M(atom=geometry, basis='STO-3G', verbose=0)
mf = scf.RHF(mol).run(verbose=0)

# FCI 精确基准
cisolver = fci.FCI(mf)
cisolver.nroots = 4
E_fcis, fcivec = cisolver.kernel()

print("=" * 60)
print("H₂ FCI 基准能量 (STO-3G)")
print("=" * 60)
for i, e in enumerate(E_fcis):
    exc_eV = (e - E_fcis[0]) * 27.2114
    print(f"E{i} = {e:.8f} Ha  |  激发能: {exc_eV:.4f} eV")

## 3. 定义原始哈密顿量和 Hilbert 空间

In [ ]:
# 将 PySCF 分子转为 NetKet 离散算符
ha_original = nkx.operator.from_pyscf_molecule(mol)

# 转换为 JAX 兼容的算子
try:
    ha_original = ha_original.to_jax_operator()
    print("✓ 成功转换为 JAX 兼容算子")
except Exception as e:
    print(f"✗ 无法转换为 JAX 算子: {e}")

# 原始 Hilbert 空间
hi_original = nkx.hilbert.SpinOrbitalFermions(
    n_orbitals=2,
    s=1/2,
    n_fermions_per_spin=(1, 1),
)

# NES-VMC 参数
K = 2  # 需要计算的态数（基态 + 1 个激发态）
n_spin_orbitals = hi_original.size

# 扩展 Hilbert 空间：K 个副本的直积
hi_extended = hi_original ** K

print(f"\n原始 Hilbert 空间大小: {hi_original.size}")
print(f"扩展 Hilbert 空间大小: {hi_extended.size}")

## 4. 定义 TotalAnsatz（行列式形式的波函数）

In [ ]:
class SingleStateAnsatz(nnx.Module):
    """单态 Ansatz"""
    def __init__(self, n_spin_orbitals: int, hidden_dim: int = 16, *, rngs: nnx.Rngs):
        super().__init__()
        self.linear1 = nnx.Linear(n_spin_orbitals, hidden_dim, rngs=rngs, param_dtype=complex)
        self.linear2 = nnx.Linear(hidden_dim, hidden_dim, rngs=rngs, param_dtype=complex)
        self.output = nnx.Linear(hidden_dim, 1, rngs=rngs, param_dtype=complex)

    def __call__(self, x):
        h = nnx.tanh(self.linear1(x.astype(complex)))
        h = nnx.tanh(self.linear2(h))
        out = self.output(h)
        return jnp.squeeze(out)


class TotalAnsatz(nnx.Module):
    """NES 总 Ansatz: 输入 K 个副本状态，返回 det[ψ_j(x^i)]"""
    def __init__(self, n_spin_orbitals: int, n_states: int = K, hidden_dim: int = 16):
        super().__init__()
        self.n_states = n_states
        self.n_spin_orbitals = n_spin_orbitals

        key = jax.random.key(42)
        keys = jax.random.split(key, n_states)

        self.single_ansatz_list = [
            SingleStateAnsatz(
                n_spin_orbitals,
                hidden_dim,
                rngs=nnx.Rngs(keys[i])
            )
            for i in range(n_states)
        ]

    def __call__(self, x_batch):
        """
        x_batch: (n_states, n_spin_orbitals) -> (psi_total, M)
        M_{ij} = ψ_j(x^i)
        """
        K_state = self.n_states
        M = []
        for i in range(K_state):
            row = [self.single_ansatz_list[j](x_batch[i]) for j in range(K_state)]
            M.append(jnp.array(row))
        M = jnp.stack(M)
        psi_total = jnp.linalg.det(M)
        return psi_total, M


# 实例化 TotalAnsatz
total_ansatz = TotalAnsatz(
    n_spin_orbitals=n_spin_orbitals,
    n_states=K,
    hidden_dim=16,
)

graph_def, params = nnx.split(total_ansatz)
print(f"✓ 创建 TotalAnsatz")
print(f"参数数量: {sum(p.size for p in jax.tree.leaves(params))}")

## 5. 构建扩展哈密顿量（矩阵形式）

In [ ]:
def build_extended_hamiltonian_matrix(hi_original, hi_extended, original_hamiltonian, K):
    """构建扩展哈密顿量的矩阵表示"""
    states_original = hi_original.all_states()
    n_states_original = states_original.shape[0]
    
    # 构建原始哈密顿量的矩阵表示
    H_original = np.zeros((n_states_original, n_states_original), dtype=complex)
    
    for i, state in enumerate(states_original):
        conn_states, mels = original_hamiltonian.get_conn(state)
        for conn_state, mel in zip(conn_states, mels):
            j = hi_original.states_to_numbers(conn_state)
            H_original[i, j] = mel
    
    # 构建扩展系统的哈密顿量
    I = np.eye(n_states_original, dtype=complex)
    H_extended = np.zeros((hi_extended.n_states, hi_extended.n_states), dtype=complex)
    
    for i in range(K):
        term = np.array([[1.0]], dtype=complex)
        for j in range(K):
            if j == i:
                term = np.kron(term, H_original)
            else:
                term = np.kron(term, I)
        H_extended = H_extended + term
    
    return H_original, H_extended


# 构建扩展哈密顿量矩阵
H_original, H_extended = build_extended_hamiltonian_matrix(
    hi_original, hi_extended, ha_original, K
)

print(f"原始哈密顿量矩阵形状: {H_original.shape}")
print(f"扩展哈密顿量矩阵形状: {H_extended.shape}")

## 6. 定义局部能量计算函数

In [ ]:
def compute_local_energy_matrix(total_ansatz, x_batch, H_extended, hi_extended, eps=1e-6):
    """
    计算扩展系统的局部能量矩阵
    
    参数:
        total_ansatz: TotalAnsatz 模型
        x_batch: (K, n_spin_orbitals) 的组态
        H_extended: 扩展哈密顿量矩阵
        hi_extended: 扩展 Hilbert 空间
        eps: 正则化参数
    
    返回:
        E_loc: 局部能量（标量）
        E_matrix: 局部能量矩阵 (K, K)
    """
    # 计算波函数值和矩阵 M
    psi_total, M = total_ansatz(x_batch)
    
    # 正则化矩阵 M
    M_reg = M + eps * jnp.eye(M.shape[0], dtype=M.dtype)
    
    # 将组态展平并找到在扩展 Hilbert 空间中的索引
    x_flat = x_batch.flatten()
    state_idx = hi_extended.states_to_numbers(x_flat)
    
    # 获取扩展哈密顿量矩阵的对应行
    H_row = H_extended[state_idx, :]
    
    # 计算所有连接态的波函数值
    # 找到所有非零矩阵元对应的连接态
    nonzero_indices = np.where(np.abs(H_row) > 1e-10)[0]
    
    # 计算 H * Psi
    H_Psi = jnp.zeros((K, K), dtype=complex)
    
    for idx in nonzero_indices:
        # 获取连接态
        conn_state = hi_extended.numbers_to_states(idx)
        conn_state_reshaped = conn_state.reshape(K, n_spin_orbitals)
        
        # 计算连接态的波函数值
        psi_conn, M_conn = total_ansatz(conn_state_reshaped)
        
        # 累加到 H_Psi
        mel = H_row[idx]
        H_Psi = H_Psi + mel * M_conn
    
    # 求解线性方程组: M * E_matrix = H_Psi
    E_matrix = jnp.linalg.solve(M_reg, H_Psi)
    
    # 局部能量是矩阵的迹
    E_loc = jnp.trace(E_matrix)
    
    return E_loc, E_matrix

## 7. 定义损失函数和梯度

In [ ]:
def loss_fn(params, H_extended, hi_extended, x_batch):
    """
    损失函数：局部能量矩阵的迹
    
    参数:
        params: 模型参数 (graph_def, variables)
        H_extended: 扩展哈密顿量矩阵
        hi_extended: 扩展 Hilbert 空间
        x_batch: (n_samples, K, n_spin_orbitals) 的样本
    """
    graph_def, variables = params
    model = nnx.merge(graph_def, variables)
    
    # 批量计算局部能量
    def single_loss(x):
        E_loc, _ = compute_local_energy_matrix(model, x, H_extended, hi_extended)
        return E_loc.real
    
    # 对所有样本计算平均
    E_locs = jax.vmap(single_loss)(x_batch)
    
    return E_locs.mean()

In [ ]:
# 创建梯度函数
value_and_grad = jax.value_and_grad(loss_fn, argnums=0)

## 8. 定义采样器

In [ ]:
def forward(params, x_batch):
    """
    采样器接口：返回 log|Ψ|
    
    参数:
        params: (graph_def, variables)
        x_batch: (n_chains, K * n_spin_orbitals)
    """
    graph_def, variables = params
    model = nnx.merge(graph_def, variables)
    
    n_chains = x_batch.shape[0]
    x_reshaped = x_batch.reshape(n_chains, K, n_spin_orbitals)
    
    def single_logpsi(x):
        psi, _ = model(x)
        return jnp.log(jnp.abs(psi) + 1e-12)
    
    log_psi_batch = jax.vmap(single_logpsi)(x_reshaped)
    return log_psi_batch

In [ ]:
# 创建采样器
sampler = nk.sampler.MetropolisLocal(
    hi_extended,
    n_chains=16,
)

# 初始化采样器状态
sampler_state = sampler.init_state(forward, (graph_def, params), seed=1)

print(f"✓ 创建采样器: {sampler}")

## 9. 训练循环

In [ ]:
# 优化器
optimizer = optax.chain(
    optax.clip_by_global_norm(1.0),
    optax.adam(learning_rate=0.01)
)
opt_state = optimizer.init(params)

# 训练参数
n_iter = 100
chain_length = 200
loss_record = []

print("\n" + "=" * 60)
print("开始训练扩展系统...")
print("=" * 60)

for step in tqdm(range(n_iter)):
    # 重置采样器并采样
    sampler_state = sampler.reset(forward, (graph_def, params), sampler_state)
    samples, sampler_state = sampler.sample(
        forward, (graph_def, params), state=sampler_state, chain_length=chain_length
    )
    
    # 重塑样本
    samples_flat = samples.reshape(-1, K, n_spin_orbitals)
    
    # 计算损失和梯度
    loss_val, grads = value_and_grad(
        (graph_def, params), H_extended, hi_extended, samples_flat
    )
    loss_record.append(loss_val)
    
    # 更新参数
    grad_graph, grad_vars = grads
    updates, opt_state = optimizer.update(grad_vars, opt_state, params)
    params = optax.apply_updates(params, updates)
    
    # 打印日志
    if step % 20 == 0:
        print(f"Step {step:4d} | Loss = {loss_val:.6f} Ha")

# 合并最终模型
total_ansatz = nnx.merge(graph_def, params)
print("\n✓ 训练完成")

## 10. 采样计算局域能量矩阵

In [ ]:
# 最终采样
print("\n" + "=" * 60)
print("最终采样，计算局域能量矩阵...")
print("=" * 60)

final_samples, _ = sampler.sample(
    forward, (graph_def, params), state=sampler_state, chain_length=1000
)
final_samples_flat = final_samples.reshape(-1, K, n_spin_orbitals)

print(f"样本数: {final_samples_flat.shape[0]}")

In [ ]:
# 计算平均局域能量矩阵
def compute_average_energy_matrix(total_ansatz, samples, H_extended, hi_extended):
    """
    计算平均局域能量矩阵
    
    参数:
        total_ansatz: TotalAnsatz 模型
        samples: (n_samples, K, n_spin_orbitals) 的样本
        H_extended: 扩展哈密顿量矩阵
        hi_extended: 扩展 Hilbert 空间
    
    返回:
        E_avg: 平均局部能量
        E_matrix_avg: 平均局域能量矩阵
    """
    def single_E_matrix(x):
        _, E_matrix = compute_local_energy_matrix(total_ansatz, x, H_extended, hi_extended)
        return E_matrix
    
    # 批量计算
    E_matrices = jax.vmap(single_E_matrix)(samples)
    
    # 平均
    E_matrix_avg = E_matrices.mean(axis=0)
    E_avg = jnp.trace(E_matrix_avg)
    
    return E_avg, E_matrix_avg


# 计算平均局域能量矩阵
E_avg, E_matrix_avg = compute_average_energy_matrix(
    total_ansatz, final_samples_flat, H_extended, hi_extended
)

print(f"\n平均局部能量: {E_avg:.6f} Ha")
print(f"\n平均局域能量矩阵:")
print(E_matrix_avg)

## 11. 对角化得到激发态能量

In [ ]:
# 对称化矩阵
E_matrix_sym = (E_matrix_avg + E_matrix_avg.conj().T) / 2

# 对角化
eigenvalues_vmc = jnp.linalg.eigvalsh(E_matrix_sym)
eigenvalues_vmc = jnp.sort(eigenvalues_vmc)

print("\n" + "=" * 60)
print("NES-VMC 计算得到的激发态能量")
print("=" * 60)
for i, e in enumerate(eigenvalues_vmc):
    exc = (e - eigenvalues_vmc[0]) * 27.2114
    print(f"E{i} = {e:.8f} Ha  |  激发能: {exc:.4f} eV")

## 12. 与 FCI 结果对比

In [ ]:
print("\n" + "=" * 60)
print("结果对比")
print("=" * 60)

print("\nVMC 结果:")
for i, e in enumerate(eigenvalues_vmc):
    error = abs(e - E_fcis[i])
    print(f"E{i} = {e:.8f} Ha  |  误差: {error:.6f} Ha")

print("\nFCI 基准:")
for i, e in enumerate(E_fcis[:K]):
    exc_eV = (e - E_fcis[0]) * 27.2114
    print(f"E{i} = {e:.8f} Ha  |  激发能: {exc_eV:.4f} eV")

## 13. 总结

本 notebook 实现了完整的 NES-VMC 流程：

1. **TotalAnsatz**：使用行列式形式的波函数
2. **扩展哈密顿量**：构建扩展系统的哈密顿量矩阵
3. **训练**：使用 MCState + VMC 训练扩展系统的基态
4. **局域能量矩阵**：采样计算平均局域能量矩阵
5. **对角化**：得到原系统的激发态能量

### 关键点

- 扩展系统的基态波函数包含了原系统多个态的信息
- 局域能量矩阵的对角化可以得到激发态能量
- 这种方法避免了直接处理扩展系统的复杂性